# LeetCode 548 — Split Array with Equal Sum bằng Python

Link đề: https://leetcode.com/problems/split-array-with-equal-sum/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Độ khó trên LeetCode: Hard; theo danh mục của repo: Medium

Ghi chú (bài Premium): đề dưới đây được viết theo hiểu biết về bài, nên đối chiếu lại với đề gốc nếu bạn có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def splitArray(self, nums: List[int]) -> bool:
        ...
```

Ghi chú: bài này là sự kết hợp của **prefix sum** (tổng tiền tố) và **cố định một vị trí cắt ở giữa + dùng `set`** để biến bốn vòng lặp lồng nhau thành hai vòng. Ta đi từ vét cạn `O(n^4)` → `O(n^3)` → `O(n^2)` và so sánh với một cách khác dùng bảng băm (dict) từ giá trị tiền tố sang các chỉ số.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm ba vị trí cắt `i < j < k` (cách nhau ít nhất một phần tử) sao cho **bốn đoạn còn lại** có tổng bằng nhau; ba phần tử bị cắt **không** tính vào tổng.
2. Nắm các công cụ Python: **prefix sum** (`s[r+1] - s[l]` là tổng đoạn `[l, r]`), `set` (thêm, kiểm tra `in`), `dict` với `setdefault`, vòng `for` với `range(a, b)` và cách tính biên chính xác.
3. Đi từ **vét cạn bốn vòng lặp** → **prefix sum `O(n^3)`** → **cố định `j` + `set` `O(n^2)`** → **bảng băm nhảy theo giá trị tiền tố**.
4. Hiểu mấu chốt: điều kiện đoạn trái (hai đoạn đầu) chỉ phụ thuộc cặp `(i, j)`, điều kiện đoạn phải (hai đoạn cuối) chỉ phụ thuộc cặp `(j, k)`; hai nửa chỉ "gặp nhau" qua **một giá trị chung** — nên lưu nửa trái vào `set` rồi tra nửa phải.
5. Tránh các lỗi kinh điển: tính cả phần tử cắt vào tổng, quên xóa `set` khi đổi `j`, chặn biên sai (cho phép đoạn rỗng), bỏ sót một điều kiện bằng nhau.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` độ dài `n`. Trả về `true` nếu tồn tại bộ ba `(i, j, k)` thỏa:

- `0 < i`, `i + 1 < j`, `j + 1 < k`, `k < n - 1`
- Tổng của các đoạn `(0, i-1)`, `(i+1, j-1)`, `(j+1, k-1)`, `(k+1, n-1)` **bằng nhau**.

Ở đây đoạn `(l, r)` là lát cắt của mảng gốc từ chỉ số `l` đến chỉ số `r` (gồm cả hai đầu). Các phần tử ở vị trí `i`, `j`, `k` là **phần tử cắt**, không thuộc đoạn nào.

**Ví dụ 1**

- Input: `nums = [1,2,1,2,1,2,1]`
- Output: `true`
- Giải thích: `i = 1, j = 3, k = 5`. Các đoạn: `(0,0) = 1`, `(2,2) = 1`, `(4,4) = 1`, `(6,6) = 1`.

**Ví dụ 2**

- Input: `nums = [1,2,1,2,1,2,1,2]`
- Output: `false`

**Ràng buộc:**

- `n == nums.length`
- `1 <= n <= 2000`
- `-10^6 <= nums[i] <= 10^6`

Hệ quả: mỗi đoạn phải có **ít nhất một phần tử** (vì `0 < i`, `i + 1 < j`, ...) và có ba phần tử cắt, nên cần `n >= 7`. Nếu `n < 7` đáp án luôn là `false`. Giá trị **có thể âm hoặc bằng 0**, nên không thể dùng cách "tăng dần tổng" kiểu hai con trỏ.

In [1]:
from typing import List
from itertools import combinations
import random
import time

ex1 = [1, 2, 1, 2, 1, 2, 1]
ex2 = [1, 2, 1, 2, 1, 2, 1, 2]

def show_split(nums, i, j, k):
    n = len(nums)
    segs = [nums[0:i], nums[i + 1:j], nums[j + 1:k], nums[k + 1:n]]
    print("cắt tại", (i, j, k), "-> phần tử cắt", [nums[i], nums[j], nums[k]])
    for name, seg in zip("ABCD", segs):
        print(f"  đoạn {name} = {seg} tổng {sum(seg)}")

print("n =", len(ex1), "| kỳ vọng True")
show_split(ex1, 1, 3, 5)
print("n =", len(ex2), "| kỳ vọng False")

n = 7 | kỳ vọng True
cắt tại (1, 3, 5) -> phần tử cắt [2, 2, 2]
  đoạn A = [1] tổng 1
  đoạn B = [1] tổng 1
  đoạn C = [1] tổng 1
  đoạn D = [1] tổng 1
n = 8 | kỳ vọng False


## 3. Kiến thức nền

### 3.1. Lát cắt (slicing) và tổng đoạn bằng `sum`

`nums[a:b]` lấy các phần tử từ chỉ số `a` đến `b - 1` (**không** gồm `b`). Muốn lấy đoạn gồm cả hai đầu `(l, r)` ta viết `nums[l:r+1]`. `sum(...)` cộng các phần tử.

Cẩn thận với lệch 1 đơn vị: đề viết `(0, i-1)` nghĩa là `nums[0:i]`; đoạn `(i+1, j-1)` nghĩa là `nums[i+1:j]`.

In [2]:
a = [5, 1, 4, 2, 8]
print(a[1:3])            # [1, 4]  (chỉ số 1 và 2)
print(a[:2], a[3:])      # [5, 1] [2, 8]
l, r = 1, 3
print("đoạn (1, 3) gồm cả hai đầu:", a[l:r + 1], "tổng", sum(a[l:r + 1]))
print("đoạn rỗng:", a[2:2], "tổng", sum(a[2:2]))

[1, 4]
[5, 1] [2, 8]
đoạn (1, 3) gồm cả hai đầu: [1, 4, 2] tổng 7
đoạn rỗng: [] tổng 0


### 3.2. Prefix sum (tổng tiền tố)

Muốn hỏi "tổng đoạn `[l, r]`" nhiều lần mà mỗi lần không phải cộng lại từ đầu, ta tính trước mảng `s` có `n + 1` phần tử:

```
s[0] = 0
s[t + 1] = s[t] + nums[t]
```

Khi đó `s[t]` là tổng của `t` phần tử đầu (`nums[0..t-1]`), và **tổng đoạn `[l, r]` = `s[r + 1] - s[l]`**. Mỗi truy vấn chỉ tốn `O(1)`.

Với bài này, bốn đoạn có công thức (đọc kỹ, đây là chỗ dễ nhầm nhất):

| Đoạn | Chỉ số | Tổng theo `s` |
|---|---|---|
| A | `(0, i-1)` | `s[i] - s[0] = s[i]` |
| B | `(i+1, j-1)` | `s[j] - s[i+1]` |
| C | `(j+1, k-1)` | `s[k] - s[j+1]` |
| D | `(k+1, n-1)` | `s[n] - s[k+1]` |

In [3]:
def prefix_sums(nums):
    s = [0] * (len(nums) + 1)
    for t, v in enumerate(nums):
        s[t + 1] = s[t] + v
    return s


s = prefix_sums(ex1)
print("nums =", ex1)
print("s    =", s)
# đoạn B của (i, j, k) = (1, 3, 5): chỉ số 2..2
print("tổng đoạn (2, 2) =", s[3] - s[2], "| kiểm tra bằng sum:", sum(ex1[2:3]))
# so sánh bốn đoạn với (1, 3, 5)
n = len(ex1)
i, j, k = 1, 3, 5
print([s[i], s[j] - s[i + 1], s[k] - s[j + 1], s[n] - s[k + 1]])   # [1, 1, 1, 1]

nums = [1, 2, 1, 2, 1, 2, 1]
s    = [0, 1, 3, 4, 6, 7, 9, 10]
tổng đoạn (2, 2) = 1 | kiểm tra bằng sum: 1
[1, 1, 1, 1]


### 3.3. `set`: thêm và kiểm tra thuộc

`set` là tập hợp không trùng lặp. `x in st` kiểm tra trong thời gian `O(1)` trung bình; `st.add(x)` thêm một phần tử. Ta dùng `set` để nhớ "những tổng đã thấy ở nửa trái".

In [4]:
seen = set()
seen.add(3)
seen.add(3)       # thêm trùng không làm tăng kích thước
seen.add(-1)
print(seen, len(seen))
print(3 in seen, 7 in seen)
seen.clear()      # xóa sạch (dùng lại cùng một set)
print(seen)

{3, -1} 2
True False
set()


### 3.4. `dict` với `setdefault`: từ giá trị đến danh sách chỉ số

Cách 4 sẽ cần bảng "giá trị tiền tố → các chỉ số có giá trị đó". `d.setdefault(key, [])` trả về list đã có (hoặc tạo list rỗng nếu chưa có) rồi ta `append` chỉ số vào đó.

In [5]:
s = prefix_sums([0, 0, 1, 0, 0])
pos = {}
for idx, val in enumerate(s):
    pos.setdefault(val, []).append(idx)
print("s   =", s)
print("pos =", pos)         # {0: [0, 1, 2], 1: [3, 4, 5]}
print(pos.get(2, []))       # giá trị không có -> []

s   = [0, 0, 0, 1, 1, 1]
pos = {0: [0, 1, 2], 1: [3, 4, 5]}
[]


### 3.5. Vòng `for` với `range(a, b)` và cách tính biên

`range(a, b)` đi `a, a+1, ..., b-1`. Ta cần giải điều kiện `0 < i`, `i+1 < j`, `j+1 < k`, `k < n-1` để chọn biên đúng cho từng vòng:

- `i >= 1`.
- `j >= i + 2`, nên `j >= 3`.
- `k >= j + 2`.
- `k <= n - 2`, kéo theo `j <= k - 2 <= n - 4`.

Vậy `j` chạy trong `range(3, n - 3)` (tức `3..n-4`), `i` chạy `range(1, j - 1)` (tức `1..j-2`), `k` chạy `range(j + 2, n - 1)` (tức `j+2..n-2`). Cell dưới kiểm chứng bằng cách liệt kê mọi bộ ba hợp lệ cho vài `n`.

In [6]:
def all_triplets(n):
    return [(i, j, k) for i in range(n) for j in range(n) for k in range(n)
            if 0 < i and i + 1 < j and j + 1 < k and k < n - 1]


for n in (5, 6, 7, 8, 9, 10):
    ts = all_triplets(n)
    js = sorted({t[1] for t in ts})
    formula = list(range(3, n - 3))
    print(f"n={n:2d}: số bộ ba = {len(ts):3d}, j thực tế = {js}, range(3, n-3) = {formula}")
    assert js == formula

n= 5: số bộ ba =   0, j thực tế = [], range(3, n-3) = []
n= 6: số bộ ba =   0, j thực tế = [], range(3, n-3) = []
n= 7: số bộ ba =   1, j thực tế = [3], range(3, n-3) = [3]
n= 8: số bộ ba =   4, j thực tế = [3, 4], range(3, n-3) = [3, 4]
n= 9: số bộ ba =  10, j thực tế = [3, 4, 5], range(3, n-3) = [3, 4, 5]
n=10: số bộ ba =  20, j thực tế = [3, 4, 5, 6], range(3, n-3) = [3, 4, 5, 6]


## 4. Ý tưởng nền tảng / trực giác

Hãy làm tay ví dụ 1: `nums = [1,2,1,2,1,2,1]`, `n = 7`.

```
chỉ số:   0  1  2  3  4  5  6
giá trị:  1  2  1  2  1  2  1
          A  x  B  x  C  x  D        (x = phần tử cắt)
```

Với `n = 7` chỉ có đúng **một** cách cắt hợp lệ: `(i, j, k) = (1, 3, 5)` — mỗi đoạn đúng 1 phần tử, ba phần tử cắt nằm xen kẽ. Bốn tổng `1, 1, 1, 1` bằng nhau nên đáp án `True`.

Ví dụ 2: `[1,2,1,2,1,2,1,2]`, `n = 8`. Giờ có nhiều cách cắt hơn, nhưng thử hết (cell bên dưới) không cách nào cho bốn tổng bằng nhau. Chẳng hạn `(1,3,5)` cho các đoạn `1, 1, 1` rồi đoạn D là `nums[6:8] = [1, 2]` có tổng 3, khác 1.

Cách người ta làm tự nhiên: **thử mọi bộ ba `(i, j, k)`** và tính tổng bốn đoạn. Ta sẽ viết đúng như thế trước (cách 1), rồi tìm cách bớt việc.

In [7]:
n = len(ex2)
for (i, j, k) in all_triplets(n):
    segs = [sum(ex2[:i]), sum(ex2[i + 1:j]), sum(ex2[j + 1:k]), sum(ex2[k + 1:])]
    print((i, j, k), segs, "<- bằng nhau" if len(set(segs)) == 1 else "")

(1, 3, 5) [1, 1, 1, 3] 
(1, 3, 6) [1, 1, 3, 2] 
(1, 4, 6) [1, 3, 2, 2] 
(2, 4, 6) [3, 2, 2, 2] 


## 5. Cách 1 — Brute Force: bốn tổng bằng `sum` trên lát cắt

### Ý tưởng

Ba vòng `for` chọn `i, j, k` đúng biên; với mỗi bộ ba, tính bốn tổng bằng `sum(nums[...])` rồi so sánh. Dễ nhất để chắc chắn đúng, nên ta cũng dùng làm **oracle** kiểm tra các cách sau.

### Code

In [8]:
def split_brute(nums: List[int]) -> bool:
    n = len(nums)
    for i in range(1, n - 1):
        for j in range(i + 2, n - 1):
            for k in range(j + 2, n - 1):
                a = sum(nums[:i])
                if (a == sum(nums[i + 1:j]) and a == sum(nums[j + 1:k])
                        and a == sum(nums[k + 1:])):
                    return True
    return False


print(split_brute(ex1), split_brute(ex2))

True False


### Độ phức tạp và nút thắt

- Số bộ ba `(i, j, k)` cỡ `n^3 / 6`; mỗi bộ ba cộng bốn đoạn, tổng độ dài `O(n)` → **`O(n^4)` thời gian**, `O(n)` bộ nhớ cho các lát cắt. Với `n = 2000` là cỡ `10^13` — không thể.
- Nút thắt 1: tổng đoạn bị **tính đi tính lại** từ đầu. Prefix sum làm nó `O(1)`.
- Nút thắt 2: dù có prefix sum, vẫn còn ba vòng lồng nhau `O(n^3)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Prefix sum.** Mỗi tổng đoạn thành `O(1)` → còn `O(n^3)` (ba vòng).

**Bước 2 — Nhìn cấu trúc của bốn điều kiện.** Gọi `S` là tổng chung. Viết lại:

- A = `s[i]` (chỉ phụ thuộc `i`).
- B = `s[j] - s[i+1]` (phụ thuộc `i, j`).
- C = `s[k] - s[j+1]` (phụ thuộc `j, k`).
- D = `s[n] - s[k+1]` (chỉ phụ thuộc `k`).

Điều kiện `A = B` chỉ có `i` và `j`. Điều kiện `C = D` chỉ có `j` và `k`. Hai nửa này **chỉ liên hệ qua `j` và giá trị chung `S`** (vì `A = B = C = D = S`).

**Bước 3 — Cố định `j` rồi tách hai nửa.**

- Nửa trái: với mỗi `i` hợp lệ có `s[i] == s[j] - s[i+1]` (nghĩa là A = B), **ghi nhớ giá trị `s[i]` vào `set`**.
- Nửa phải: với mỗi `k` hợp lệ có `s[n] - s[k+1] == s[k] - s[j+1]` (C = D), kiểm tra giá trị chung đó có nằm trong `set` hay không.

Nếu có, tức tồn tại `i` và `k` cho cùng một `S` → bốn đoạn bằng nhau → `True`.

Thay vì thử mọi cặp `(i, k)` (`O(n^2)` cho mỗi `j`), ta quét `i` một lần và `k` một lần: mỗi `j` tốn `O(n)`, tổng `O(n^2)`.

In [9]:
# Minh họa Bước 3 trên ví dụ 1 với j = 3
nums = ex1
n = len(nums)
s = prefix_sums(nums)
j = 3
left_vals = {}
for i in range(1, j - 1):
    A, B = s[i], s[j] - s[i + 1]
    print(f"i={i}: A={A}, B={B} ->", "A == B, lưu S =", A if A == B else "bỏ")
    if A == B:
        left_vals[i] = A
for k in range(j + 2, n - 1):
    C, D = s[k] - s[j + 1], s[n] - s[k + 1]
    print(f"k={k}: C={C}, D={D} ->", "C == D, S =", C if C == D else "bỏ", "| có trong nửa trái?", C in left_vals.values())

i=1: A=1, B=1 -> A == B, lưu S = 1
k=5: C=1, D=1 -> C == D, S = 1 | có trong nửa trái? True


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) brute `O(n^4)` (mục 5), (2) prefix sum + ba vòng `O(n^3)`, (3) cố định `j` + `set` `O(n^2)`, (4) bảng băm giá trị tiền tố, nhảy `i → j → k`.

### 7.1. Cách 2 — Prefix sum + ba vòng lặp `O(n^3)`

Giải thích từng đoạn:

1. Dựng `s` bằng `prefix_sums` (`O(n)`).
2. Ba vòng `i, j, k` đúng biên.
3. Mỗi bộ ba kiểm tra bốn tổng bằng công thức ở mục 3.2 trong `O(1)`.

In [10]:
def split_prefix_triple(nums: List[int]) -> bool:
    n = len(nums)
    s = prefix_sums(nums)
    for i in range(1, n):
        a = s[i]
        for j in range(i + 2, n):
            if s[j] - s[i + 1] != a:        # B != A: bỏ luôn, khỏi xét k
                continue
            for k in range(j + 2, n - 1):
                if s[k] - s[j + 1] == a and s[n] - s[k + 1] == a:
                    return True
    return False


print(split_prefix_triple(ex1), split_prefix_triple(ex2))

True False


In [11]:
# Code nộp LeetCode cho cách 2 (đúng nhưng chậm với n = 2000 trong trường hợp xấu)
class SolutionTriple:
    def splitArray(self, nums: List[int]) -> bool:
        n = len(nums)
        s = [0] * (n + 1)
        for t, v in enumerate(nums):
            s[t + 1] = s[t] + v
        for i in range(1, n):
            for j in range(i + 2, n):
                if s[j] - s[i + 1] != s[i]:
                    continue
                for k in range(j + 2, n - 1):
                    if s[k] - s[j + 1] == s[i] and s[n] - s[k + 1] == s[i]:
                        return True
        return False


print(SolutionTriple().splitArray(ex1), SolutionTriple().splitArray(ex2))

True False


**Độ phức tạp:** `O(n^3)` tệ nhất (mảng toàn số 0 nhưng cuối cùng không thỏa — mọi `(i, j)` đều qua được bộ lọc `B == A`), `O(n)` bộ nhớ cho `s`. Với `n = 2000` vẫn quá chậm trong trường hợp xấu.

### 7.2. Cách 3 — Cố định `j` + `set` (lời giải nộp) `O(n^2)`

Giải thích từng đoạn:

1. `n < 7` thì vòng `j` rỗng, tự trả `False` (không cần xử lý riêng).
2. Với mỗi `j` trong `range(3, n - 3)`: tạo `seen = set()` **mới** (quan trọng: xóa mỗi lần đổi `j`).
3. Quét `i` trong `range(1, j - 1)`: nếu `s[i] == s[j] - s[i+1]` (A = B) thì `seen.add(s[i])`.
4. Quét `k` trong `range(j + 2, n - 1)`: nếu `s[n] - s[k+1] == s[k] - s[j+1]` (C = D) **và** giá trị đó thuộc `seen` thì trả `True`.
5. Hết mọi `j` mà không thấy → `False`.

In [12]:
def split_fix_j(nums: List[int]) -> bool:
    n = len(nums)
    s = prefix_sums(nums)
    for j in range(3, n - 3):
        seen = set()
        for i in range(1, j - 1):
            if s[i] == s[j] - s[i + 1]:
                seen.add(s[i])
        for k in range(j + 2, n - 1):
            if s[n] - s[k + 1] == s[k] - s[j + 1] and s[n] - s[k + 1] in seen:
                return True
    return False


print(split_fix_j(ex1), split_fix_j(ex2))

True False


In [13]:
# Code nộp LeetCode cho cách 3 (cố định j + set)
class SolutionFixJ:
    def splitArray(self, nums: List[int]) -> bool:
        n = len(nums)
        s = [0] * (n + 1)
        for i, v in enumerate(nums):
            s[i + 1] = s[i] + v
        for j in range(3, n - 3):
            seen = set()
            for i in range(1, j - 1):
                if s[i] == s[j] - s[i + 1]:
                    seen.add(s[i])
            for k in range(j + 2, n - 1):
                if s[n] - s[k + 1] == s[k] - s[j + 1] and s[n] - s[k + 1] in seen:
                    return True
        return False


print(SolutionFixJ().splitArray(ex1), SolutionFixJ().splitArray(ex2))

True False


**Độ phức tạp:** mỗi `j` quét `i` (`<= j`) và `k` (`<= n - j`) một lần, tổng `O(n)` mỗi `j` → **`O(n^2)` thời gian** (`4 * 10^6` bước với `n = 2000`), `O(n)` bộ nhớ (`s` và `seen`).

### 7.3. Cách 4 — Bảng băm giá trị tiền tố, nhảy `i → j → k`

Ý tưởng khác: thay vì quét mọi `j`, ta **suy ra thẳng vị trí cần tìm** từ giá trị tiền tố.

Cố định `i` và đặt `S = s[i]`. Điều kiện B: `s[j] - s[i+1] = S` ⇔ `s[j] = s[i+1] + S`. Vậy `j` phải là chỉ số có giá trị tiền tố **đúng bằng** `s[i+1] + S` — tra trong bảng `pos` (giá trị → danh sách chỉ số tăng dần). Tương tự C: `s[k] = s[j+1] + S`; và D: `s[k+1] = s[n] - S`.

Thuật toán:

1. `pos[v]` = danh sách chỉ số `t` mà `s[t] == v` (tăng dần).
2. Với mỗi `i`, `S = s[i]`: duyệt `j` trong `pos[s[i+1] + S]` thỏa `j >= i + 2` (và `j <= n - 4`).
3. Với mỗi `j`, duyệt `k` trong `pos[s[j+1] + S]` thỏa `j + 2 <= k <= n - 2` rồi kiểm tra `s[k+1] == s[n] - S`.

Nếu các giá trị tiền tố **đều khác nhau** (ví dụ mảng toàn số dương) thì mỗi danh sách có tối đa 1 chỉ số → gần như `O(n)`. Nhưng khi **rất nhiều tiền tố trùng nhau** (nhiều số 0) thì danh sách dài và tệ nhất có thể quay về `O(n^3)`.

In [14]:
def split_dict_jump(nums: List[int]) -> bool:
    n = len(nums)
    s = prefix_sums(nums)
    pos = {}
    for t, val in enumerate(s):
        pos.setdefault(val, []).append(t)
    total = s[n]
    for i in range(1, n - 5):                  # i <= n - 6 để còn chỗ cho j, k
        S = s[i]
        for j in pos.get(s[i + 1] + S, ()):
            if j < i + 2:
                continue
            if j > n - 4:
                break                          # danh sách tăng dần nên có thể dừng
            for k in pos.get(s[j + 1] + S, ()):
                if k < j + 2:
                    continue
                if k > n - 2:
                    break
                if s[k + 1] == total - S:
                    return True
    return False


print(split_dict_jump(ex1), split_dict_jump(ex2))

True False


In [15]:
# Code nộp LeetCode cho cách 4 (bảng băm nhảy; nhanh với dữ liệu "thưa", chậm nếu quá nhiều tiền tố trùng)
class SolutionDictJump:
    def splitArray(self, nums: List[int]) -> bool:
        n = len(nums)
        s = [0] * (n + 1)
        for t, v in enumerate(nums):
            s[t + 1] = s[t] + v
        pos = {}
        for t, val in enumerate(s):
            pos.setdefault(val, []).append(t)
        for i in range(1, n - 5):
            S = s[i]
            for j in pos.get(s[i + 1] + S, ()):
                if j < i + 2:
                    continue
                if j > n - 4:
                    break
                for k in pos.get(s[j + 1] + S, ()):
                    if k < j + 2:
                        continue
                    if k > n - 2:
                        break
                    if s[k + 1] == s[n] - S:
                        return True
        return False


print(SolutionDictJump().splitArray(ex1), SolutionDictJump().splitArray(ex2))

True False


**Độ phức tạp:** dựng bảng `O(n)`. Phần tìm: với dữ liệu mà các giá trị tiền tố khác nhau đôi một là `O(n)`; **tệ nhất** (rất nhiều tiền tố trùng, ví dụ gần toàn số 0) là `O(n^3)`. Bộ nhớ `O(n)`. Vì chặn trên xấu nên **ta nộp cách 3** (đảm bảo `O(n^2)`); cách 4 hữu ích để hiểu cách "tra giá trị thay vì quét", và ta sẽ đo sự chênh lệch ở mục 11.

## 8. TRACE từng bước

### 8.1. Ví dụ 1: `nums = [1,2,1,2,1,2,1]`

`n = 7`, `s = [0,1,3,4,6,7,9,10]`. Vòng `j` chạy `range(3, 4)` nên chỉ `j = 3`.

| `j` | `i` | `s[i]` | `s[j]-s[i+1]` | A=B? | `seen` |
|---|---|---|---|---|---|
| 3 | 1 | 1 | `4 - 3 = 1` | có | `{1}` |

| `j` | `k` | `s[n]-s[k+1]` (D) | `s[k]-s[j+1]` (C) | C=D? | D trong `seen`? | Kết quả |
|---|---|---|---|---|---|---|
| 3 | 5 | `10 - 9 = 1` | `7 - 6 = 1` | có | có | **True** với `(1, 3, 5)` |

### 8.2. Ví dụ 2: `nums = [1,2,1,2,1,2,1,2]`

`n = 8`, `s = [0,1,3,4,6,7,9,10,12]`, `j` chạy `3, 4`.

| `j` | nửa trái (`i`: A, B) | `seen` | nửa phải (`k`: C, D) | Kết luận |
|---|---|---|---|---|
| 3 | `i=1`: A=1, B=1 (lưu 1) | `{1}` | `k=5`: C=1, D=3 (khác); `k=6`: C=3, D=2 (khác) | không có |
| 4 | `i=1`: A=1, B=3 (bỏ); `i=2`: A=3, B=2 (bỏ) | `{}` | `k=6`: C=2, D=2 (bằng) nhưng `seen` rỗng | không có |

Hết `j` → **False**. Hàm `trace_split` in lại đúng các bước này.

In [16]:
def trace_split(nums):
    n = len(nums)
    s = prefix_sums(nums)
    print("n =", n, "| s =", s)
    if n < 7:
        print("n < 7 -> chắc chắn False")
        return False
    for j in range(3, n - 3):
        print(f"--- j = {j} ---")
        seen = set()
        for i in range(1, j - 1):
            A, B = s[i], s[j] - s[i + 1]
            flag = "A == B -> thêm vào seen" if A == B else "khác -> bỏ"
            if A == B:
                seen.add(A)
            print(f"  i={i}: A=s[{i}]={A}, B=s[{j}]-s[{i + 1}]={B}  {flag}")
        print("  seen =", seen)
        for k in range(j + 2, n - 1):
            C, D = s[k] - s[j + 1], s[n] - s[k + 1]
            note = ""
            if C == D:
                note = "C == D" + (" và nằm trong seen -> TÌM THẤY" if D in seen else ", nhưng không nằm trong seen")
            print(f"  k={k}: C=s[{k}]-s[{j + 1}]={C}, D=s[{n}]-s[{k + 1}]={D}  {note}")
            if C == D and D in seen:
                print("  => True với j =", j, "k =", k)
                return True
    print("hết j -> False")
    return False


print(trace_split(ex1))
print()
print(trace_split(ex2))

n = 7 | s = [0, 1, 3, 4, 6, 7, 9, 10]
--- j = 3 ---
  i=1: A=s[1]=1, B=s[3]-s[2]=1  A == B -> thêm vào seen
  seen = {1}
  k=5: C=s[5]-s[4]=1, D=s[7]-s[6]=1  C == D và nằm trong seen -> TÌM THẤY
  => True với j = 3 k = 5
True

n = 8 | s = [0, 1, 3, 4, 6, 7, 9, 10, 12]
--- j = 3 ---
  i=1: A=s[1]=1, B=s[3]-s[2]=1  A == B -> thêm vào seen
  seen = {1}
  k=5: C=s[5]-s[4]=1, D=s[8]-s[6]=3  
  k=6: C=s[6]-s[4]=3, D=s[8]-s[7]=2  
--- j = 4 ---
  i=1: A=s[1]=1, B=s[4]-s[2]=3  khác -> bỏ
  i=2: A=s[2]=3, B=s[4]-s[3]=2  khác -> bỏ
  seen = set()
  k=6: C=s[6]-s[5]=2, D=s[8]-s[7]=2  C == D, nhưng không nằm trong seen
hết j -> False
False


### 8.3. Một ví dụ có số âm và số 0

Để thấy thuật toán không cần giả sử gì về dấu, ta dựng sẵn một mảng có bốn đoạn cùng tổng `S = 0` nhờ số âm, xen ba phần tử cắt `7`, `8`, `9`: `[1,-1] | 7 | [2,-2] | 8 | [0] | 9 | [3,-3]`. Cell dưới chạy `trace_split` trên đó.

In [17]:
# Bốn đoạn tổng 0: [1,-1] | [2,-2] | [0] | [3,-3], xen các phần tử cắt 7, 8, 9
nums_neg = [1, -1, 7, 2, -2, 8, 0, 9, 3, -3]
print("kiểm tra bằng brute:", split_brute(nums_neg))
trace_split(nums_neg)

kiểm tra bằng brute: True
n = 10 | s = [0, 1, 0, 7, 9, 7, 15, 15, 24, 27, 24]
--- j = 3 ---
  i=1: A=s[1]=1, B=s[3]-s[2]=7  khác -> bỏ
  seen = set()
  k=5: C=s[5]-s[4]=-2, D=s[10]-s[6]=9  
  k=6: C=s[6]-s[4]=6, D=s[10]-s[7]=9  
  k=7: C=s[7]-s[4]=6, D=s[10]-s[8]=0  
  k=8: C=s[8]-s[4]=15, D=s[10]-s[9]=-3  
--- j = 4 ---
  i=1: A=s[1]=1, B=s[4]-s[2]=9  khác -> bỏ
  i=2: A=s[2]=0, B=s[4]-s[3]=2  khác -> bỏ
  seen = set()
  k=6: C=s[6]-s[5]=8, D=s[10]-s[7]=9  
  k=7: C=s[7]-s[5]=8, D=s[10]-s[8]=0  
  k=8: C=s[8]-s[5]=17, D=s[10]-s[9]=-3  
--- j = 5 ---
  i=1: A=s[1]=1, B=s[5]-s[2]=7  khác -> bỏ
  i=2: A=s[2]=0, B=s[5]-s[3]=0  A == B -> thêm vào seen
  i=3: A=s[3]=7, B=s[5]-s[4]=-2  khác -> bỏ
  seen = {0}
  k=7: C=s[7]-s[6]=0, D=s[10]-s[8]=0  C == D và nằm trong seen -> TÌM THẤY
  => True với j = 5 k = 7


True

## 9. Code nộp LeetCode

Chọn cách 3 (cố định `j` + `set`) làm lời giải nộp: đảm bảo `O(n^2)`, ngắn gọn.

In [18]:
class Solution:
    def splitArray(self, nums: List[int]) -> bool:
        n = len(nums)
        s = [0] * (n + 1)
        for i, v in enumerate(nums):
            s[i + 1] = s[i] + v
        for j in range(3, n - 3):
            seen = set()
            for i in range(1, j - 1):
                if s[i] == s[j] - s[i + 1]:
                    seen.add(s[i])
            for k in range(j + 2, n - 1):
                if s[n] - s[k + 1] == s[k] - s[j + 1] and s[n] - s[k + 1] in seen:
                    return True
        return False


sol = Solution()
print(sol.splitArray([1, 2, 1, 2, 1, 2, 1]))      # True
print(sol.splitArray([1, 2, 1, 2, 1, 2, 1, 2]))   # False

True
False


## 10. Vì sao đúng

**Bước 1: Đúng đề.** Một bộ ba `(i, j, k)` hợp lệ khi `1 <= i`, `i + 2 <= j`, `j + 2 <= k`, `k <= n - 2` và bốn tổng bằng nhau:

`s[i] = s[j] - s[i+1] = s[k] - s[j+1] = s[n] - s[k+1]`.

Theo mục 3.5, `j` chỉ có thể nằm trong `3..n-4` (vòng ngoài `range(3, n-3)` không bỏ sót `j` nào). Với `j` cố định, `i` chạy `1..j-2` và `k` chạy `j+2..n-2` (đúng các vòng trong code).

**Bước 2: Hai chiều của "tương đương".**

- *Nếu code trả `True`*: có `i` với `s[i] == s[j] - s[i+1]` và `s[i] = S` được lưu; có `k` với `s[n]-s[k+1] = s[k]-s[j+1] = S`. Vậy `s[i] = s[j]-s[i+1] = S = s[k]-s[j+1] = s[n]-s[k+1]` — bốn tổng bằng `S`.
- *Nếu tồn tại bộ ba hợp lệ `(i, j, k)` với tổng chung `S`*: khi vòng ngoài tới đúng `j` đó, `i` thỏa `s[i] = s[j]-s[i+1] = S` nên `S` được thêm vào `seen`; rồi `k` thỏa C = D = `S` và `S in seen`, nên code trả `True`.

**Bước 3: Dùng `set` không mất thông tin.** Chỉ có giá trị `S` mới nối nửa trái với nửa phải (A và D phải bằng nhau, B và C cũng thế), và vị trí `i`, `k` độc lập nhau với mỗi `j` cố định (miền `i < j`, `k > j`). Nên chỉ cần biết **tập các giá trị `S`** nửa trái, không cần biết `i`.

**Bước 4: Số âm và số 0.** Lập luận chỉ dùng đẳng thức của prefix sum, **không** giả sử đơn điệu, nên vẫn đúng với số âm/0 (đó cũng là lý do ta không dùng hai con trỏ).

In [19]:
# Kiểm chứng "mọi bộ ba hợp lệ nằm trong miền quét của code" bằng cách đếm
def count_triplets_in_code_domain(n):
    cnt = 0
    for j in range(3, n - 3):
        for i in range(1, j - 1):
            for k in range(j + 2, n - 1):
                cnt += 1
    return cnt


for n in range(1, 16):
    assert count_triplets_in_code_domain(n) == len(all_triplets(n)), n
print("miền (i, j, k) mà code quét == tập mọi bộ ba hợp lệ của đề, với n = 1..15")

miền (i, j, k) mà code quét == tập mọi bộ ba hợp lệ của đề, với n = 1..15


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute `sum` | `O(n^4)` | `O(n)` |
| 2. Prefix sum + 3 vòng | `O(n^3)` | `O(n)` |
| 3. Cố định `j` + `set` (nộp) | `O(n^2)` | `O(n)` |
| 4. Bảng băm tiền tố | `O(n)` ... `O(n^3)` tùy dữ liệu | `O(n)` |

Với `n = 2000`: cách 3 cỡ `4 * 10^6` thao tác — chạy được trong khoảng một giây ở Python. Cell dưới đếm số lần ta kiểm tra điều kiện để thấy tăng trưởng bậc 2 (gấp đôi `n` thì số bước gấp ~4 lần) trên trường hợp xấu cho cách 3 (mảng toàn số 1 → luôn `False`).

In [20]:
def count_steps_fix_j(nums):
    n = len(nums)
    s = prefix_sums(nums)
    steps = 0
    for j in range(3, n - 3):
        seen = set()
        for i in range(1, j - 1):
            steps += 1
            if s[i] == s[j] - s[i + 1]:
                seen.add(s[i])
        for k in range(j + 2, n - 1):
            steps += 1
            if s[n] - s[k + 1] == s[k] - s[j + 1] and s[n] - s[k + 1] in seen:
                return steps, True
    return steps, False


prev = None
for n in (100, 200, 400, 800):
    steps, ans = count_steps_fix_j([1] * n)
    ratio = "" if prev is None else f"(x{steps / prev:.2f} so với n/2)"
    print(f"n={n:4d}: {steps:7d} bước, đáp án {ans} {ratio}")
    prev = steps

n= 100:    8930 bước, đáp án False 
n= 200:   37830 bước, đáp án False (x4.24 so với n/2)
n= 400:  155630 bước, đáp án False (x4.11 so với n/2)


n= 800:  631230 bước, đáp án False (x4.06 so với n/2)


In [21]:
# So sánh thời gian thực tế trên mảng toàn số 1 và trên mảng "rất nhiều số 0, một số 1 ở cuối"
def timed(f, nums):
    t0 = time.perf_counter()
    r = f(nums)
    return r, time.perf_counter() - t0


n = 2000
ones = [1] * n
print("n = 2000, mảng toàn số 1 (False):")
for name, f in [("cố định j + set", Solution().splitArray),
                ("bảng băm nhảy", SolutionDictJump().splitArray)]:
    r, dt = timed(f, ones)
    print(f"  {name:16s}: {r}  {dt * 1000:8.1f} ms")

n = 120
zeros1 = [0] * (n - 1) + [1]
print("n = 120, toàn số 0 trừ số cuối là 1 (False, nhiều tiền tố trùng):")
for name, f in [("cố định j + set", Solution().splitArray),
                ("bảng băm nhảy", SolutionDictJump().splitArray),
                ("prefix 3 vòng", SolutionTriple().splitArray)]:
    r, dt = timed(f, zeros1)
    print(f"  {name:16s}: {r}  {dt * 1000:8.1f} ms")

n = 2000, mảng toàn số 1 (False):


  cố định j + set : False     300.2 ms
  bảng băm nhảy   : False       1.1 ms
n = 120, toàn số 0 trừ số cuối là 1 (False, nhiều tiền tố trùng):
  cố định j + set : False       0.7 ms
  bảng băm nhảy   : False      33.3 ms
  prefix 3 vòng   : False      18.5 ms


## 12. Bộ test

Chiến lược: (a) ví dụ đề; (b) edge case (n nhỏ, đúng 7, 0, âm, biên giá trị); (c) phản ví dụ cho các cách sai; (d) **vét cạn** mọi mảng nhỏ với giá trị `{-1, 0, 1}` và đối chiếu với brute; (e) ngẫu nhiên có seed, gồm cả ca **dựng sẵn đáp án True**; (f) kích thước lớn nhất.

In [22]:
solvers = {
    "prefix 3 vòng": split_prefix_triple,
    "cố định j": split_fix_j,
    "bảng băm": split_dict_jump,
    "nộp Solution": lambda a: Solution().splitArray(a),
    "nộp Triple": lambda a: SolutionTriple().splitArray(a),
    "nộp FixJ": lambda a: SolutionFixJ().splitArray(a),
    "nộp DictJump": lambda a: SolutionDictJump().splitArray(a),
}


def check_all(nums, expected, label=""):
    assert split_brute(nums) == expected, ("brute", label, nums)
    for name, f in solvers.items():
        got = f(list(nums))
        assert got == expected, (label, name, got, expected, nums)


# (a) ví dụ đề
check_all(ex1, True, "ví dụ 1")
check_all(ex2, False, "ví dụ 2")
print("ví dụ đề: OK")

ví dụ đề: OK


In [23]:
# (b) edge case
for n in range(1, 7):
    check_all([0] * n, False, f"n={n} < 7")
    check_all(list(range(1, n + 1)), False, f"n={n} tăng dần")
check_all([0] * 7, True, "bảy số 0")
check_all([5] * 7, True, "bảy số 5 (đoạn đơn có tổng 5)")
check_all([1, 5, 1, 7, 1, 9, 1], True, "cắt bằng số khác")
check_all([1, 5, 1, 7, 1, 9, 2], False, "đoạn cuối khác")
check_all([0, 0, 0, 0, 0, 0, 1], False, "chỉ có 1 ở cuối")
check_all([1, 0, 0, 0, 0, 0, 0], False, "chỉ có 1 ở đầu")
check_all([1, -1, 7, 2, -2, 8, 0, 9, 3, -3], True, "bốn đoạn tổng 0 có số âm")
big = 10 ** 6
check_all([big, 3, big, 3, big, 3, big], True, "biên +10^6")
check_all([-big, 3, -big, 3, -big, 3, -big], True, "biên -10^6")
check_all([big, 3, big, 3, big, 3, -big], False, "biên lệch dấu")
print("edge case: OK")

edge case: OK


In [24]:
# (c) phản ví dụ: những cách làm sai (ở mục 13 sẽ phân tích) cho kết quả sai trên đúng các input này
def wrong_includes_cut(nums):
    """SAI: coi phần tử cắt thuộc đoạn đứng trước nó (thực chất chia mảng thành 4 đoạn liền nhau)."""
    n = len(nums)
    s = prefix_sums(nums)
    for j in range(3, n - 3):
        seen = set()
        for i in range(1, j - 1):
            if s[i + 1] == s[j + 1] - s[i + 1]:          # A = nums[0..i] (nuốt phần tử cắt i)
                seen.add(s[i + 1])
        for k in range(j + 2, n - 1):
            C, D = s[k + 1] - s[j + 1], s[n] - s[k + 1]   # C = nums[j+1..k], D = nums[k+1..]
            if C == D and D in seen:
                return True
    return False


print("ví dụ 1 đúng True, bản sai trả:", wrong_includes_cut(ex1))
assert wrong_includes_cut(ex1) != True

ví dụ 1 đúng True, bản sai trả: False


In [25]:
# (d) vét cạn: mọi mảng độ dài 7..9 với giá trị trong {-1, 0, 1}
from itertools import product

cnt = 0
cnt_true = 0
for n in (7, 8, 9):
    for tup in product((-1, 0, 1), repeat=n):
        nums = list(tup)
        expected = split_brute(nums)
        for name, f in solvers.items():
            assert f(list(nums)) == expected, (name, nums)
        cnt += 1
        cnt_true += expected
print("đã đối chiếu", cnt, "mảng, trong đó True:", cnt_true, ": OK")

đã đối chiếu 28431 mảng, trong đó True: 4351 : OK


In [26]:
# (e1) random có seed: giá trị nhỏ để nhiều tổng trùng nhau (kích thích nhánh set)
rng = random.Random(548)
n_true = 0
for case in range(1500):
    n = rng.randint(1, 16)
    lo = rng.choice([1, 2, 3])
    nums = [rng.randint(-lo, lo) for _ in range(n)]
    expected = split_brute(nums)
    n_true += expected
    check_all(nums, expected, f"random case {case}")
print("1500 ca ngẫu nhiên nhỏ: OK (số ca True:", n_true, ")")

1500 ca ngẫu nhiên nhỏ: OK (số ca True: 164 )


In [27]:
# (e2) dựng sẵn đáp án True: chọn tổng chung S, bốn đoạn dài >= 1 cùng tổng S, ba phần tử cắt tùy ý
def segment_with_sum(rng, length, S, lim=50):
    vals = [rng.randint(-lim, lim) for _ in range(length - 1)]
    vals.append(S - sum(vals))
    rng.shuffle(vals)
    return vals


def planted_true(rng, max_len=6, lim=50):
    S = rng.randint(-100, 100)
    parts = [segment_with_sum(rng, rng.randint(1, max_len), S, lim) for _ in range(4)]
    cuts = [rng.randint(-lim, lim) for _ in range(3)]
    return parts[0] + [cuts[0]] + parts[1] + [cuts[1]] + parts[2] + [cuts[2]] + parts[3]


rng = random.Random(99)
for case in range(300):
    nums = planted_true(rng)
    assert Solution().splitArray(nums) is True, nums
    assert SolutionDictJump().splitArray(nums) is True, nums
    assert split_prefix_triple(nums) is True
print("300 ca dựng sẵn True: OK")

# (e3) phá một phần tử rồi hỏi lại: đáp án mới không biết trước, đối chiếu với brute độc lập
changed = 0
for case in range(600):
    nums = planted_true(rng, max_len=3, lim=5)
    p = rng.randrange(len(nums))
    nums[p] += rng.choice([-2, -1, 1, 2])
    expected = split_brute(nums)
    changed += (not expected)
    check_all(nums, expected, f"perturbed {case}")
print("600 ca bị phá: OK (số ca chuyển thành False:", changed, ")")

300 ca dựng sẵn True: OK
600 ca bị phá: OK (số ca chuyển thành False: 432 )


In [28]:
# (f) cỡ lớn nhất n = 2000 (ràng buộc |nums[i]| <= 10^6)
rng = random.Random(2000)
n = 2000

cases = {
    "toàn số 0 (True)": ([0] * n, True),
    "toàn số 1 (False)": ([1] * n, False),
    "số ngẫu nhiên lớn (False gần như chắc)": ([rng.randint(-10 ** 6, 10 ** 6) for _ in range(n)], None),
}
for name, (nums, expected) in cases.items():
    t0 = time.perf_counter()
    got = Solution().splitArray(nums)
    dt = time.perf_counter() - t0
    if expected is not None:
        assert got == expected
    print(f"{name:42s} -> {got}  ({dt * 1000:.0f} ms)")

# True "muộn": đoạn B dài m phần tử 1 nên j rất lớn, buộc quét gần hết các j
m = n - 6
nums = [m, 0] + [1] * m + [0, m, 0, m]       # i = 1, j = m + 2, k = m + 4
assert len(nums) == n
t0 = time.perf_counter()
got = Solution().splitArray(nums)
assert got is True
print("True muộn:", got, f"({(time.perf_counter() - t0) * 1000:.0f} ms)")

toàn số 0 (True)                           -> True  (0 ms)


toàn số 1 (False)                          -> False  (299 ms)


số ngẫu nhiên lớn (False gần như chắc)     -> False  (306 ms)


True muộn: True (298 ms)


## 13. Các lỗi người mới hay gặp

Để các lỗi "có thật" thay vì đoán, ta viết hàm `find_counterexample(wrong_fn)` quét mảng nhỏ ngẫu nhiên (seed cố định) và so với `split_brute`, in mảng đầu tiên mà bản sai trả sai.

In [29]:
def find_counterexample(wrong_fn, trials=20000, seed=13):
    rng = random.Random(seed)
    for _ in range(trials):
        n = rng.randint(7, 12)
        nums = [rng.randint(-2, 2) for _ in range(n)]
        if wrong_fn(nums) != split_brute(nums):
            return nums
    return None

### Lỗi 1 — Tính cả phần tử cắt vào tổng (lệch chỉ số)

Đề nói phần tử `i`, `j`, `k` **không** thuộc đoạn nào, nên đoạn A là `s[i]` (tổng `nums[0..i-1]`), không phải `s[i+1]`. Bản sai ở mục 12(c) đã dịch chỉ số lệch 1.

In [30]:
ce = find_counterexample(wrong_includes_cut)
print("phản ví dụ:", ce)
print("đúng (brute) =", split_brute(ce), "| bản sai =", wrong_includes_cut(ce))
print("ví dụ đề: đúng =", split_brute(ex1), "| bản sai =", wrong_includes_cut(ex1))

phản ví dụ: [0, 0, -1, 0, 2, -1, 2, -1, 1, -1, 2, -2]
đúng (brute) = True | bản sai = False
ví dụ đề: đúng = True | bản sai = False


**Vì sao sai:** nếu tính `s[i+1]` thì đoạn A thành `nums[0..i]` (nuốt luôn phần tử cắt `nums[i]`), và các đoạn khác cũng lệch theo. Kết quả sai ngay trên ví dụ 1. **Đúng:** A = `s[i]`, B = `s[j] - s[i+1]`, C = `s[k] - s[j+1]`, D = `s[n] - s[k+1]`.

### Lỗi 2 — Không xóa `seen` khi đổi `j`

Đặt `seen = set()` **ngoài** vòng `for j` khiến các tổng của `j` trước còn sót lại ở `j` sau, dù chúng chưa chắc hợp lệ với `j` mới.

In [31]:
def wrong_seen_not_reset(nums):
    n = len(nums)
    s = prefix_sums(nums)
    seen = set()                           # SAI: đặt ngoài vòng j
    for j in range(3, n - 3):
        for i in range(1, j - 1):
            if s[i] == s[j] - s[i + 1]:
                seen.add(s[i])
        for k in range(j + 2, n - 1):
            if s[n] - s[k + 1] == s[k] - s[j + 1] and s[n] - s[k + 1] in seen:
                return True
    return False


ce = find_counterexample(wrong_seen_not_reset)
print("phản ví dụ:", ce)
print("đúng (brute) =", split_brute(ce), "| bản sai =", wrong_seen_not_reset(ce))

phản ví dụ: [1, 2, 2, -1, 1, 1, 2, -1, 0, -1, -1, 2]
đúng (brute) = False | bản sai = True


**Vì sao sai:** một `S` thu được ở `j = 3` nhờ cặp `(i, 3)`; sang `j = 4`, ta kiểm tra `k` với `seen` còn chứa `S` cũ nhưng điều kiện A = B **đã không được kiểm cho `j = 4`**, nên có thể xác nhận bộ ba không có thật. **Đúng:** tạo `seen = set()` **bên trong** vòng `for j`.

### Lỗi 3 — Quên một điều kiện bằng nhau (chỉ so A với D)

Bốn đoạn bằng nhau cần **ba** đẳng thức độc lập (A = B, C = D, và A = C hoặc nối hai nửa qua `set`). Nếu ở nửa phải chỉ kiểm `D in seen` mà bỏ điều kiện `C == D`, ta không bảo đảm đoạn C.

In [32]:
def wrong_missing_cd(nums):
    n = len(nums)
    s = prefix_sums(nums)
    for j in range(3, n - 3):
        seen = set()
        for i in range(1, j - 1):
            if s[i] == s[j] - s[i + 1]:
                seen.add(s[i])
        for k in range(j + 2, n - 1):
            if s[n] - s[k + 1] in seen:          # SAI: thiếu s[k] - s[j+1] == s[n] - s[k+1]
                return True
    return False


ce = find_counterexample(wrong_missing_cd)
print("phản ví dụ:", ce)
print("đúng (brute) =", split_brute(ce), "| bản sai =", wrong_missing_cd(ce))

phản ví dụ: [2, 2, 1, 0, 1, 0, -2, 0, 2, 0]
đúng (brute) = False | bản sai = True


**Vì sao sai:** thiếu điều kiện `C == D` nên đoạn giữa phải `(j+1..k-1)` có tổng tùy ý. **Đúng:** kiểm cả `s[n]-s[k+1] == s[k]-s[j+1]` rồi mới tra `seen`.

### Lỗi 4 — Chặn biên cho phép đoạn rỗng

Nếu cho `i` chạy `range(1, j)` (thay vì `range(1, j - 1)`), ta có `i = j - 1`, khi đó đoạn B `(i+1, j-1)` **rỗng** (tổng 0) — đề không cho phép vì cần `i + 1 < j`.

In [33]:
def wrong_allows_empty(nums):
    n = len(nums)
    s = prefix_sums(nums)
    for j in range(3, n - 3):
        seen = set()
        for i in range(1, j):                      # SAI: nên là range(1, j - 1)
            if s[i] == s[j] - s[i + 1]:
                seen.add(s[i])
        for k in range(j + 2, n - 1):
            if s[n] - s[k + 1] == s[k] - s[j + 1] and s[n] - s[k + 1] in seen:
                return True
    return False


ce = find_counterexample(wrong_allows_empty)
print("phản ví dụ:", ce)
print("đúng (brute) =", split_brute(ce), "| bản sai =", wrong_allows_empty(ce))

phản ví dụ: [2, -2, 0, 2, 2, 0, -1, 0, -1, -1, 2]
đúng (brute) = False | bản sai = True


**Vì sao sai:** với `i = j - 1`, B là đoạn rỗng có tổng 0 và nếu `s[i] = 0` thì `A = B = 0` được ghi nhận, dù đoạn B "không tồn tại". **Đúng:** tính biên từ đúng điều kiện đề: `i` chạy `1..j-2`, `k` chạy `j+2..n-2`.

### Lỗi 5 — Dùng hai con trỏ / tham lam vì tưởng mọi số dương

Với số dương, tổng tiền tố tăng ngặt nên có thể "đi tới" khi tổng đủ. Nhưng đề cho số **âm** và **0**, tổng tiền tố không đơn điệu — tham lam sẽ bỏ sót lời giải. Ví dụ đề thật sự có đáp án `True` mà mọi tổng tiền tố không tăng:

In [34]:
nums = [1, -1, 7, 2, -2, 8, 0, 9, 3, -3]
print("tiền tố:", prefix_sums(nums))
print("đáp án đúng:", split_fix_j(nums))


def wrong_greedy_positive(nums):
    """SAI: coi đoạn A chỉ là nums[0], rồi mỗi đoạn kế tiếp cắt NGAY khi tổng chạm S (chỉ hợp lý khi toàn dương)."""
    n = len(nums)
    S, idx = nums[0], 1
    for _ in range(3):
        idx += 1                              # bỏ phần tử cắt
        total = 0
        while idx < n and total < S:
            total += nums[idx]
            idx += 1
        if total != S:
            return False
    return idx == n


print("tham lam trả:", wrong_greedy_positive(nums))
assert wrong_greedy_positive(nums) != split_fix_j(nums)

tiền tố: [0, 1, 0, 7, 9, 7, 15, 15, 24, 27, 24]
đáp án đúng: True
tham lam trả: False


**Vì sao sai:** khi có số âm, một đoạn có thể "chạm" `S` rồi vẫn tiếp tục thêm các số cộng lại bằng 0; tham lam cắt sớm hoặc không cắt được. **Đúng:** dùng prefix sum + `set`/bảng băm, vốn không cần tính đơn điệu.

## 14. Mẹo & nhận dạng pattern

- Đề có "chia mảng thành các đoạn có tổng bằng nhau" + "tổng đoạn bất kỳ" → nghĩ ngay **prefix sum**.
- Nhiều vòng lặp lồng nhau mà điều kiện **tách được thành hai nửa chỉ chung một giá trị** → **cố định phần ở giữa**, lưu nửa này vào `set`/`dict`, tra nửa kia. (Cùng họ với Two Sum, 4Sum, "meet in the middle".)
- Luôn đặt phần tử cố định là phần tử **giữa** (`j`) để nửa trái và nửa phải độc lập.
- Khi giá trị có thể âm hoặc 0, đừng dùng hai con trỏ/cửa sổ trượt dựa trên đơn điệu.
- Cẩn thận biên: viết sẵn ba bất đẳng thức của đề, giải ra miền của từng chỉ số (mục 3.5) rồi hãy viết `range`.
- Tự kiểm chứng bằng **brute O(n^4)** trên mảng nhỏ — rẻ và bắt được mọi lỗi lệch biên.
- Mẹo cắt tỉa nhanh: `n < 7` trả `False` ngay; tổng cả mảng `s[n]` phải bằng `4S + nums[i] + nums[j] + nums[k]` nên có thể lọc thêm nếu muốn.

## 15. Biến thể

### 15.1. Đếm số bộ ba `(i, j, k)` hợp lệ

Thay `set` bằng `Counter` (đếm số lần mỗi `S` xuất hiện ở nửa trái), rồi cộng `Counter[S]` cho mỗi `k` thỏa C = D.

In [35]:
from collections import Counter


def count_triplets(nums):
    n = len(nums)
    s = prefix_sums(nums)
    total = 0
    for j in range(3, n - 3):
        left = Counter()
        for i in range(1, j - 1):
            if s[i] == s[j] - s[i + 1]:
                left[s[i]] += 1
        for k in range(j + 2, n - 1):
            D = s[n] - s[k + 1]
            if D == s[k] - s[j + 1]:
                total += left[D]
    return total


def count_triplets_brute(nums):
    n = len(nums)
    cnt = 0
    for (i, j, k) in all_triplets(n):
        a = sum(nums[:i])
        cnt += (a == sum(nums[i + 1:j]) == sum(nums[j + 1:k]) == sum(nums[k + 1:]))
    return cnt


print(count_triplets(ex1), count_triplets([0] * 9), count_triplets_brute([0] * 9))
rng = random.Random(5)
for _ in range(300):
    arr = [rng.randint(-1, 1) for _ in range(rng.randint(1, 11))]
    assert count_triplets(arr) == count_triplets_brute(arr)
print("đếm bộ ba khớp brute: OK")

1 10 10
đếm bộ ba khớp brute: OK


### 15.2. Trả về chính bộ ba `(i, j, k)` (hoặc `None`)

Lưu `dict` từ `S` sang một chỉ số `i` thay vì `set`.

In [36]:
def find_triplet(nums):
    n = len(nums)
    s = prefix_sums(nums)
    for j in range(3, n - 3):
        seen = {}                               # S -> một chỉ số i
        for i in range(1, j - 1):
            if s[i] == s[j] - s[i + 1]:
                seen[s[i]] = i
        for k in range(j + 2, n - 1):
            D = s[n] - s[k + 1]
            if D == s[k] - s[j + 1] and D in seen:
                return seen[D], j, k
    return None


print(find_triplet(ex1), find_triplet(ex2))
i, j, k = find_triplet(ex1)
show_split(ex1, i, j, k)

(1, 3, 5) None
cắt tại (1, 3, 5) -> phần tử cắt [2, 2, 2]
  đoạn A = [1] tổng 1
  đoạn B = [1] tổng 1
  đoạn C = [1] tổng 1
  đoạn D = [1] tổng 1


### 15.3. Chia thành **3** đoạn có tổng bằng nhau (LeetCode 1013 — Partition Array Into Three Parts With Equal Sum)

Chỉ có hai nhát cắt và **không bỏ** phần tử nào; chỉ cần `total % 3 == 0` và đếm khi chạm `total / 3`.

In [37]:
def can_three_parts(arr):
    total = sum(arr)
    if total % 3:
        return False
    target, acc, parts = total // 3, 0, 0
    for v in arr:
        acc += v
        if acc == target:
            parts += 1
            acc = 0
    return parts >= 3


print(can_three_parts([0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1]))   # True
print(can_three_parts([0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1]))   # False
print(can_three_parts([3, 3, 6, 5, -2, 2, 5, 1, -9, 4]))      # True

True
False
True


### 15.4. Các hướng mở rộng khác

- Tổng quát hóa: chia thành `m` đoạn (bỏ `m - 1` phần tử cắt) — khi `m` lớn cần DP hoặc kỹ thuật khác.
- Chỉ cần tổng bằng nhau nhưng **không bỏ phần tử cắt**: đếm số cách (LeetCode 1712 *Ways to Split Array Into Three Subarrays*, dùng hai con trỏ vì số không âm).
- Cùng họ ý tưởng "cố định giữa, băm hai bên": 4Sum II (LeetCode 454), Two Sum.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute `sum` | Thử mọi `(i, j, k)`, cộng bốn lát cắt | `O(n^4)` | `O(n)` | Làm oracle kiểm tra, `n` rất nhỏ |
| 2. Prefix sum + 3 vòng | Tổng đoạn `O(1)`, lọc sớm `A != B` | `O(n^3)` | `O(n)` | `n` vài chục, dễ viết, ít sai |
| 3. Cố định `j` + `set` | Tách nửa trái/phải qua giá trị chung `S` | `O(n^2)` | `O(n)` | **Lời giải nộp**, mọi dữ liệu |
| 4. Bảng băm tiền tố | Tra thẳng `j`, `k` từ giá trị `s[·]` | `O(n)` đến `O(n^3)` | `O(n)` | Dữ liệu ít trùng tiền tố; để hiểu "tra thay vì quét" |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output rồi chạy để kiểm tra:

```python
print(prefix_sums([2, -1, 3, 0]))
print(find_triplet([0, 1, 0, 1, 0, 1, 0]))
print(Solution().splitArray([1, 1, 1, 1, 1, 1]))
```

In [38]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [1, 1, 9, 2, 0, 8, 1, 1, 7, 2, 0]` (n = 11), hãy tính `s`, rồi với `j = 5` hãy liệt kê các `i` có A = B (kèm `S`) và các `k` có C = D (kèm `S`). Đáp án `splitArray` của mảng này là gì và bộ ba nào cho ra nó?

In [39]:
# Bài 2: điền kết quả tính tay
hand_s = None            # list 12 số
hand_left_S = None       # dict {i: S} cho j = 5
hand_right_S = None      # dict {k: S} cho j = 5
hand_answer = None
hand_triplet = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm tính tổng đoạn `[l, r]` bằng prefix sum và hàm kiểm tra một bộ ba cụ thể.

In [40]:
def exercise_3_range_sum(s: List[int], l: int, r: int) -> int:
    # TODO 1: trả về tổng nums[l..r] dựa vào mảng prefix sum s
    return None


def exercise_3_is_valid_triplet(nums: List[int], i: int, j: int, k: int) -> bool:
    n = len(nums)
    s = prefix_sums(nums)
    # TODO 2: kiểm tra 0 < i, i+1 < j, j+1 < k, k < n-1
    # TODO 3: kiểm tra bốn tổng đoạn bằng nhau, dùng exercise_3_range_sum
    return None


# assert exercise_3_range_sum(prefix_sums([4, 1, 2, 3]), 1, 2) == 3
# assert exercise_3_is_valid_triplet(ex1, 1, 3, 5) is True
# assert exercise_3_is_valid_triplet(ex1, 1, 2, 5) is False       # vi phạm i + 1 < j (1 + 1 < 2 sai)
# assert exercise_3_is_valid_triplet(ex2, 1, 3, 5) is False

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 3, hãy tự viết `exercise_4_split(nums)`: dựng prefix sum, **cố định `j`**, dùng `set` cho nửa trái, quét nửa phải.

In [41]:
def exercise_4_split(nums: List[int]) -> bool:
    # TODO
    pass


# assert exercise_4_split(ex1) is True
# assert exercise_4_split(ex2) is False
# assert exercise_4_split([0] * 7) is True
# assert exercise_4_split([1, -1, 7, 2, -2, 8, 0, 9, 3, -3]) is True

### Bài 5 — Mini challenge: đếm số bộ ba (khó)

Viết `exercise_5_count(nums)` trả về **số bộ ba `(i, j, k)`** thỏa đề. Gợi ý: thay `set` bằng đếm (`Counter` hoặc `dict`).

```python
exercise_5_count([1, 2, 1, 2, 1, 2, 1]) -> 1
exercise_5_count([0] * 8) -> 4
exercise_5_count([1, 2, 1, 2, 1, 2, 1, 2]) -> 0
```

In [42]:
def exercise_5_count(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_count(ex1) == 1
# assert exercise_5_count([0] * 8) == 4
# assert exercise_5_count(ex2) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `s[0] = 0`, mỗi phần tử cộng dồn.
- Mức 2: `find_triplet` trả bộ ba đầu tiên tìm thấy, theo thứ tự vòng `j` rồi `k`.
- Mức 3: `[0, 2, 1, 4, 4]`, `(1, 3, 5)`, `False` (n = 6 < 7).

**Bài 2**
- Mức 1: `s = [0, 1, 2, 11, 13, ...]`, cộng dồn.
- Mức 2: Với `j = 5`: `i` chạy `1..3`; A = `s[i]`, B = `s[5] - s[i+1]`.
- Mức 3: Với `j = 5`: chỉ `i = 2` thỏa (A = B = 2); chỉ `k = 8` thỏa (C = D = 2). Xem đáp án mục 19.

**Bài 3**
- Mức 1: Tổng `nums[l..r]` = `s[r + 1] - s[l]`.
- Mức 2: Bốn đoạn: `[0, i-1]`, `[i+1, j-1]`, `[j+1, k-1]`, `[k+1, n-1]`.
- Mức 3: So sánh cả bốn bằng `len({a, b, c, d}) == 1`.

**Bài 4**
- Mức 1: `for j in range(3, n - 3)`.
- Mức 2: `seen = set()` bên trong vòng `j`; quét `i` trong `range(1, j - 1)` rồi quét `k` trong `range(j + 2, n - 1)`.
- Mức 3: Xem đáp án mục 19.

**Bài 5**
- Mức 1: Một `S` ở nửa trái có thể đến từ nhiều `i`.
- Mức 2: Dùng `Counter`; mỗi `k` thỏa C = D cộng `Counter[D]`.
- Mức 3: Xem `count_triplets` ở mục 15.1.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [43]:
print(prefix_sums([2, -1, 3, 0]))                          # [0, 2, 1, 4, 4]
print(find_triplet([0, 1, 0, 1, 0, 1, 0]))                  # (1, 3, 5)
print(Solution().splitArray([1, 1, 1, 1, 1, 1]))            # False

assert prefix_sums([2, -1, 3, 0]) == [0, 2, 1, 4, 4]
assert find_triplet([0, 1, 0, 1, 0, 1, 0]) == (1, 3, 5)
assert Solution().splitArray([1, 1, 1, 1, 1, 1]) is False

[0, 2, 1, 4, 4]
(1, 3, 5)
False


### Đáp án Bài 2

Mảng gồm bốn đoạn `[1,1] | 9 | [2,0] | 8 | [1,1] | 7 | [2,0]` cùng tổng 2; ba phần tử cắt là `9`, `8`, `7` ở các chỉ số `i = 2`, `j = 5`, `k = 8`.

In [44]:
nums2 = [1, 1, 9, 2, 0, 8, 1, 1, 7, 2, 0]
n = len(nums2)
s2 = prefix_sums(nums2)
print("s =", s2)
j = 5
left = {i: s2[i] for i in range(1, j - 1) if s2[i] == s2[j] - s2[i + 1]}
right = {k: s2[n] - s2[k + 1] for k in range(j + 2, n - 1) if s2[n] - s2[k + 1] == s2[k] - s2[j + 1]}
print("j = 5: nửa trái {i: S} =", left, "| nửa phải {k: S} =", right)
print("đáp án splitArray =", Solution().splitArray(nums2), "| bộ ba =", find_triplet(nums2))
assert s2 == [0, 1, 2, 11, 13, 13, 21, 22, 23, 30, 32, 32]
assert left == {2: 2} and right == {8: 2}
assert Solution().splitArray(nums2) is True and find_triplet(nums2) == (2, 5, 8)
assert split_brute(nums2) is True

s = [0, 1, 2, 11, 13, 13, 21, 22, 23, 30, 32, 32]
j = 5: nửa trái {i: S} = {2: 2} | nửa phải {k: S} = {8: 2}
đáp án splitArray = True | bộ ba = (2, 5, 8)


### Đáp án Bài 3

In [45]:
def exercise_3_range_sum_solution(s: List[int], l: int, r: int) -> int:
    return s[r + 1] - s[l]


def exercise_3_is_valid_triplet_solution(nums: List[int], i: int, j: int, k: int) -> bool:
    n = len(nums)
    s = prefix_sums(nums)
    if not (0 < i and i + 1 < j and j + 1 < k and k < n - 1):
        return False
    a = exercise_3_range_sum_solution(s, 0, i - 1)
    b = exercise_3_range_sum_solution(s, i + 1, j - 1)
    c = exercise_3_range_sum_solution(s, j + 1, k - 1)
    d = exercise_3_range_sum_solution(s, k + 1, n - 1)
    return len({a, b, c, d}) == 1


assert exercise_3_range_sum_solution(prefix_sums([4, 1, 2, 3]), 1, 2) == 3
assert exercise_3_is_valid_triplet_solution(ex1, 1, 3, 5) is True
assert exercise_3_is_valid_triplet_solution(ex1, 1, 2, 5) is False
assert exercise_3_is_valid_triplet_solution(ex2, 1, 3, 5) is False
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [46]:
def exercise_4_split_solution(nums: List[int]) -> bool:
    n = len(nums)
    s = prefix_sums(nums)
    for j in range(3, n - 3):
        seen = set()
        for i in range(1, j - 1):
            if s[i] == s[j] - s[i + 1]:
                seen.add(s[i])
        for k in range(j + 2, n - 1):
            D = s[n] - s[k + 1]
            if D == s[k] - s[j + 1] and D in seen:
                return True
    return False


assert exercise_4_split_solution(ex1) is True
assert exercise_4_split_solution(ex2) is False
assert exercise_4_split_solution([0] * 7) is True
assert exercise_4_split_solution([1, -1, 7, 2, -2, 8, 0, 9, 3, -3]) is True
rng = random.Random(4)
for _ in range(300):
    arr = [rng.randint(-1, 1) for _ in range(rng.randint(1, 12))]
    assert exercise_4_split_solution(arr) == split_brute(arr)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [47]:
def exercise_5_count_solution(nums: List[int]) -> int:
    n = len(nums)
    s = prefix_sums(nums)
    total = 0
    for j in range(3, n - 3):
        left = Counter()
        for i in range(1, j - 1):
            if s[i] == s[j] - s[i + 1]:
                left[s[i]] += 1
        for k in range(j + 2, n - 1):
            D = s[n] - s[k + 1]
            if D == s[k] - s[j + 1]:
                total += left[D]
    return total


assert exercise_5_count_solution(ex1) == 1
assert exercise_5_count_solution([0] * 8) == 4
assert exercise_5_count_solution(ex2) == 0
rng = random.Random(6)
for _ in range(300):
    arr = [rng.randint(-1, 1) for _ in range(rng.randint(1, 11))]
    assert exercise_5_count_solution(arr) == count_triplets_brute(arr)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def splitArray(self, nums: List[int]) -> bool:
        n = len(nums)
        s = [0] * (n + 1)
        for i, v in enumerate(nums):
            s[i + 1] = s[i] + v
        for j in range(3, n - 3):                 # cố định điểm cắt giữa
            seen = set()                          # mới cho MỖI j
            for i in range(1, j - 1):             # nửa trái: A == B
                if s[i] == s[j] - s[i + 1]:
                    seen.add(s[i])
            for k in range(j + 2, n - 1):         # nửa phải: C == D và D đã thấy
                if s[n] - s[k + 1] == s[k] - s[j + 1] and s[n] - s[k + 1] in seen:
                    return True
        return False
```

Bốn đoạn: `A = s[i]`, `B = s[j] - s[i+1]`, `C = s[k] - s[j+1]`, `D = s[n] - s[k+1]`.

### Checklist trước khi submit

- [ ] Công thức bốn đoạn **không** chứa phần tử cắt (dùng `s[i+1]`, `s[j+1]`, `s[k+1]` đúng chỗ).
- [ ] Biên: `j` trong `range(3, n - 3)`, `i` trong `range(1, j - 1)`, `k` trong `range(j + 2, n - 1)`; mỗi đoạn có ít nhất một phần tử.
- [ ] `seen = set()` được tạo **bên trong** vòng `for j`.
- [ ] Nửa phải kiểm cả `C == D` **và** `D in seen`.
- [ ] Không dùng giả thiết số dương/đơn điệu; số 0 và số âm vẫn đúng.
- [ ] `n < 7` tự trả `False`; đã thử `n = 2000` (khoảng `4 * 10^6` bước).
- [ ] Signature đúng: `class Solution`, `splitArray(self, nums: List[int]) -> bool`.